## Pregunta 4.

####  ¿Está relacionado el hecho de pagar **comisiones más altas** con la obtención de una **rentabilidad superior**? 
---



#### Índice


1. Lectura de datos ya procesados.
2. Cálculo de rentabilidad neta acumulada y estudio de resultados.
3. Nuevo dataframe dividido según calificación de gastos.
4. Visualización de gráficas y estudio de resultados.
    - Gráficas
    - Fondo **ejemplo** y fondo **fracaso**
    


---

In [ ]:
import pandas as pd
from functools import reduce
import matplotlib.pyplot as plt
import numpy as np
from typing import Optional

### 1. Lectura de datos previamente procesados
 - Los datos que aquí tratamos provienen del módulo de limpieza de datos de fondos.

In [ ]:
df = pd.read_excel('../data/processed/Excel_final_fondos.xlsx', index_col= 0)

# Pasamos a numérico los gastos corrientes, ya que se quedan tipo object
df['Gastos Corrientes'] = pd.to_numeric(df['Gastos Corrientes'], errors='coerce')
df.head()

### 2. Cálculo de la rentabilidad neta acumulada y estudio de resultados.

 - Primero creamos una función que nos calcule la **rentabilidad neta acumulada**.

 Este resultado lo que permite es obtener la ganancia en el periodo propuesto. Una simple multiplicación de este número con el aporte inicial que hicimos nos dará como resultado el capital que hemos ganado, NO EL TOTAL. El **total** sería la **suma de lo inicial mas lo ganado**.

In [ ]:
''' Funciones para obtener rentabilidad neta y rentabilidad total '''

def rentab_neta_acumulada(row, initial_year=2017) -> float:
    '''
    Calculo rentabilidad neta anual si me lo piden como parámetro
    Calculo la rentabilidad total acumulada de un fondo desde 2017 (default) o otro año superior hasta 2024. 
    Tengo en cuenta las comisiones anuales y el incremento proporcionado por el interés compuesto.
    '''
    
    productorio = []
    for i in range(initial_year, 2025):
        # Calculo la rentabilidad neta
        rentab_neta  = (1 + row[f'rent {i}']) * (1 - row['Gastos Corrientes']) # calculo rentabilidad y resto gastos al capital bruto anual
        # Lo añado a la lista
        productorio.append(rentab_neta)

    # Multiplico los factores de crecimiento y le resto uno para quedarme con el porcentaje de crecimiento.
    producto_total = reduce(lambda x,y: x*y, productorio) - 1 
    return producto_total
   

# Aplicamos la funcion a nuestro dataframe por filas y creamos la nueva columna.
df['Rentabilidad_total_acumulada'] = df.apply(lambda row: rentab_neta_acumulada(row, 2017), axis=1)


- Ahora estudiamos la rentabilidad total obtenida.

In [ ]:
# volvemos a pasar a porcentaje la rentabilidad total acumulada y gastos (para visualización)
df[['Rentabilidad_total_acumulada', 'Gastos Corrientes']] = df[['Rentabilidad_total_acumulada', 'Gastos Corrientes']]*100
df['Rentabilidad_total_acumulada'].describe()


### 3. Nuevo dataframe dividido según calificación de gastos

- Ahora vamos a dividir la columna de Gastos Corrientes en 3 grupos: Bajo, medio y alto. Esto lo vamos a hacer gracias a los terciles, que nos dividirán por frecuencia.

In [ ]:
# Creamos nueva columna con la calificación de comision
df['calificacion_gastos'] = pd.qcut(df['Gastos Corrientes'], 3, labels=['bajo', 'medio', 'alto'])
df['calificacion_gastos'].head()


- Ahora hacemos groupby según calificación de gastos y ya tendremos nuestro dataframe dividido según la cantidad de gastos corrientes

In [ ]:
# Dataframe agrupado por gastos.
grouped_df = df.groupby('calificacion_gastos')
grouped_df['Rentabilidad_total_acumulada'].describe()


### 4. Visualización de resultados.

- Representaremos los resultados obtenidos en un **scatterplot y un boxplot**.

- Tras esto, escogeremos los identificadores del **fondo ejemplo y fondo fracaso**. (relación rentabilidad-gastos)

#### 4.1 Gráficas

In [ ]:
''' Scatterplot con matplotlib '''

color_map = {
    'bajo': 'cyan',
    'medio': 'cornflowerblue',
    'alto': 'darkblue'
}

# obtengo el axis
ax = plt.gca()

for clave, df_group in grouped_df:
    # Scatterplot
    df_group.plot(kind='scatter', x = 'Gastos Corrientes', y='Rentabilidad_total_acumulada', ax=ax,
             color = color_map[clave], label=clave, xlabel='Gastos Corrientes (%)', ylabel= 'Rentabilidad neta acumulada (%)')
    # Línea de tendencia
    coefs = np.polyfit(df_group['Gastos Corrientes'], df_group['Rentabilidad_total_acumulada'], 1)
    tendencia = np.poly1d(coefs)

    # Plot con la línea de tendencia creada
    x_fit = np.linspace(df_group['Gastos Corrientes'].min(), df_group['Gastos Corrientes'].max(), 100)  # Valores de x para la línea de tendencia
    y_fit = tendencia(x_fit)                 # Valores de y correspondientes
    ax.plot(x_fit, y_fit, color=color_map[clave])

plt.tight_layout()
plt.savefig("../reports/figures/funds/funds_fees_vs_return_scatter.png", dpi=300, bbox_inches="tight")


In [ ]:
''' Boxplots con matplotlib '''

# Crear la figura y los boxplots
fig, ax = plt.subplots(figsize=(8, 6))

# Crear boxplot para los 3 arrays
ax.boxplot([grouped_df.get_group('bajo')['Rentabilidad_total_acumulada'], grouped_df.get_group('medio')['Rentabilidad_total_acumulada'], 
            grouped_df.get_group('alto')['Rentabilidad_total_acumulada']], widths=0.6, patch_artist=True,
           boxprops=dict(facecolor='cyan'))

# Personalizar el gráfico
ax.set_xticks([1, 2, 3])  # Etiquetas en el eje X
ax.set_xticklabels(['Comision baja', 'Comision media', 'Comision alta'])
ax.set_ylabel('Rentabilidad (%)')
ax.set_title('Rentabilidades según comisión')

plt.tight_layout()
plt.savefig("../reports/figures/funds/funds_return_by_fee_boxplot.png", dpi=300, bbox_inches="tight")

#### 4.2 Fondo **ejemplo** y fondo **fracaso**

In [ ]:
''' Identificadores de fondo ejemplo y fondo fracaso '''
for clave, group in grouped_df:
    if clave == 'alto':
        rent = group['Rentabilidad_total_acumulada'].min()
        low_rentab_high_incomes_Isin = group.loc[group['Rentabilidad_total_acumulada']==rent, 'ISIN'].values[0]
        print(f'Fondo rentabilidad mala y altos gastos: {rent:.2f}%, ISIN: {low_rentab_high_incomes_Isin}')
    if clave == 'bajo':
        rent2 = group['Rentabilidad_total_acumulada'].max()
        high_rentab_low_incomes_Isin = group.loc[group['Rentabilidad_total_acumulada']==rent2, 'ISIN'].values[0]
        print(f'Fondo rentabilidad buena y bajos gastos: {rent2:.2f}%, ISIN: {high_rentab_low_incomes_Isin}')


Por último, sacamos el identificador del fondo más rentable del dataset. 

In [ ]:
df.loc[df['Rentabilidad_total_acumulada']== df['Rentabilidad_total_acumulada'].max(), 'ISIN'].values[0]